In [ ]:
import glob
import os
import shutil
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt


Load the coordinates of every point in the chosen set (e.g. one chain) from each input file and store them in a dictionary keyed by file path.

In [ ]:
def load_coords(file_path, chain_id):
    """Return a list of (x, y, z) tuples for all ATOM records on `chain_id`."""
    points = []
    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("ATOM  "):
                if line[21].strip() != chain_id:
                    continue
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                points.append((x, y, z))
    return points


# Settings
INPUT_PATTERN = "path/to/input_files/*.pdb"
QUERY_CHAIN = "B"

coords_by_file = {}

input_files = glob.glob(INPUT_PATTERN, recursive=True)
print(f"Found {len(input_files)} files")

for filepath in input_files:
    print(f"Processing file: {filepath}")
    coords_by_file[filepath] = load_coords(filepath, QUERY_CHAIN)


Define the reference region (the set of points the others are compared against).

In [ ]:
def load_reference_coords(file_path, target_pairs):
    """Return an array of (x, y, z) for atoms matching any (chain, residue number) pair."""
    ref_coords = []
    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("ATOM"):
                chain = line[21].strip()
                res_num = line[22:26].strip()

                if (chain, res_num) in target_pairs:
                    try:
                        x = float(line[30:38].strip())
                        y = float(line[38:46].strip())
                        z = float(line[46:54].strip())
                        ref_coords.append((x, y, z))
                    except ValueError:
                        continue

    result = np.array(ref_coords)
    print(f"Found {len(result)} atoms for the reference region.")
    return result


REFERENCE_CHAINS = ["A", "B", "C", "D", "E"]
REFERENCE_RESIDUES = ["49"]

target_pairs = [(c, r) for c in REFERENCE_CHAINS for r in REFERENCE_RESIDUES]

REFERENCE_FILE = "path/to/reference_file.pdb"
reference_points = load_reference_coords(REFERENCE_FILE, target_pairs)


**Stage 1 (proximity filter).** Keep a file only if, for every reference point, at least one query point lies within `threshold` distance (units of the input coordinates, here Å).

In [ ]:
def proximity_filter(coords_by_file, reference_points, threshold):
    """
    Keep files where every reference point has at least one
    query point within `threshold`.
    """
    passing = {}

    for filepath, coords in coords_by_file.items():
        if not coords:
            continue

        coords_array = np.array(coords)

        # distances: (n query points, n reference points)
        distances = cdist(coords_array, reference_points)

        # For each reference point, distance to its closest query point
        min_distances_per_reference = distances.min(axis=0)

        if np.all(min_distances_per_reference < threshold):
            passing[filepath] = coords

    return passing


for threshold in [5.0, 6.0, 7.0, 8.0, 10.0]:
    passing = proximity_filter(coords_by_file, reference_points, threshold)
    print(f"Files passing at threshold {threshold}: {len(passing)}")
    for path in passing:
        print(path)


**Stage 2 (density filter).** Apply the stage 1 filter with `threshold1`, then count the query points within `threshold2` of the reference points and compute the average count per reference point. Files are kept if that average exceeds `min_mean`.

In [ ]:
def two_stage_filter(coords_by_file, reference_points, threshold1, threshold2, min_mean=0.0):
    stage1 = proximity_filter(coords_by_file, reference_points, threshold1)

    results = {}

    for filepath, coords in stage1.items():
        if not coords:
            continue

        coords_array = np.array(coords)
        distances = cdist(coords_array, reference_points)

        n_contacts = (distances < threshold2).sum()
        mean_per_reference = n_contacts / len(reference_points)

        if mean_per_reference > min_mean:
            results[filepath] = mean_per_reference

    return results


In [ ]:
# Parameters
threshold1 = 8.0
threshold2 = 5.0

# Sweep over different minimum-mean thresholds
min_mean_values = np.arange(5, 21, 1)

passed_counts = []

for min_mean in min_mean_values:
    passing = two_stage_filter(
        coords_by_file,
        reference_points,
        threshold1,
        threshold2,
        min_mean
    )
    passed_counts.append(len(passing))


# Plot
plt.figure(figsize=(8, 5))
plt.plot(min_mean_values, passed_counts, marker="o")
plt.xlabel("Minimum mean contacts per reference point")
plt.ylabel("Files passed")
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

scores = two_stage_filter(coords_by_file, reference_points, threshold1, threshold2, min_mean=0.0)
for path, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"{score:.2f}  {os.path.basename(path)}")


In [ ]:
min_mean = 10

passing = two_stage_filter(
    coords_by_file,
    reference_points,
    threshold1,
    threshold2,
    min_mean
)

print(f"Passing files: {len(passing)}")

output_dir = "passed"
os.makedirs(output_dir, exist_ok=True)

for path in passing:
    print(path)
    shutil.copy2(path, os.path.join(output_dir, os.path.basename(path)))
